In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
from config.root_path import DATA_ROOT,WEIGHT_ROOT
from urllib.request import urlopen
from PIL import Image
from models.maxvit_flash import maxvit_t_flash
from models.model import get_model
from modules.train_utils import DataHandler
from tqdm import tqdm
import time
device = torch.device('cuda:0')
from torchsummary import summary

In [26]:
momodel = maxvit_t_flash(
    input_size=(224, 224),
    num_classes=8
)

In [2]:
model_name = 'maxvit'
result_folder='maxvit'

In [3]:
model = get_model(model_name,8)

/home/wangcheng/anaconda3/envs/sd/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


*******maxvit loaded *******


In [4]:
state_dict = torch.load(os.path.join(WEIGHT_ROOT,result_folder,'best_loss.pth'))
from collections import OrderedDict

# 移除键名中的 'module.' 前缀
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    if k.startswith('module.module'):
        
        new_state_dict[k[14:]] = v
    elif k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

In [5]:
set(model.state_dict().keys()) == set(new_state_dict.keys())

True

In [6]:
model.load_state_dict(new_state_dict)
model.to(device)
sum(p.numel() for p in model.parameters())

MaxVit(
  (stem): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
    (1): Conv2dNormActivation(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
  )
  (blocks): ModuleList(
    (0): MaxVitBlock(
      (layers): ModuleList(
        (0): MaxVitLayer(
          (layers): Sequential(
            (MBconv): MBConv(
              (proj): Sequential(
                (0): AvgPool2d(kernel_size=3, stride=2, padding=1)
                (1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
              )
              (stochastic_depth): Identity()
              (layers): Sequential(
                (pre_norm): BatchNorm2d(64, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
                (conv_a): Conv2dNormActivation(
           

In [7]:
net = nn.DataParallel(model,device_ids=list( range(8) ) )

In [8]:
data_loader = DataHandler(os.path.join(DATA_ROOT,'8_class_select'),256,8,model_name,result_folder)
test_loader ,test_dataset= data_loader.test_loader,data_loader.test_dataset
class_to_idx = test_dataset.class_to_idx

In [9]:
sum(p.numel() for p in model.parameters())

30411728

In [10]:
correct = 0
total = 0
top3_correct = 0
start_time = time.time()
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        _, top3_predicted = outputs.topk(3, 1, True, True)
        
        total += labels.size(0)
        correct += (predicted == labels.to(device)).sum().item()
        
        # Check if the true label is in the top 3 predictions
        for i in range(labels.size(0)):
            if labels[i] in top3_predicted[i]:
                top3_correct += 1

accuracy = 100 * correct / total
top3_accuracy = 100 * top3_correct / total
end_time = time.time()
elapsed_time = end_time - start_time

print(f'Accuracy: {accuracy:.2f}%')
print(f'Top-3 Accuracy: {top3_accuracy:.2f}%')
print(f'Elapsed Time: {elapsed_time:.2f} seconds')
        

100%|██████████| 10/10 [00:16<00:00,  1.64s/it]

Accuracy: 18.93%
Top-3 Accuracy: 50.02%
Elapsed Time: 16.39 seconds


In [ ]:
x = torch.random()

In [ ]:
import torch
model = maxvit_custom(
    input_size=(224, 224),
    num_classes=8
)
x = torch.randn(2, 3, 224, 224).cuda().half()
model = model.cuda().half()
output = model(x)
print(f"Output shape: {output.shape}")  # Should be [2, 8]

In [ ]:
import flash_attn
import torch
capability = torch.cuda.get_device_capability()
        # 检查是否为支持的GPU架构
capability[0]